# Windshield Detection — YOLOv8n Hyperparameter Grid Search  

**Objective**:  Find the best hyperparameter configuration using grid search with cross-validation.

**Dataset**: The `train/` and `val/` sets are merged into a single training pool. The `test/` set is kept separate and will only be used for the final evaluation.

**Strategy**: Each hyperparameter configuration is evaluated using 3-fold cross-validation. The results are averaged across folds to get a more reliable estimate of performance.

---

### Pipeline

1. Merge `train` and `val` sets into a single data pool  
2. Split the dataset into 3 folds  
3. For each hyperparameter configuration and each fold:  
   - Train a YOLO model (short runs with early stopping)  
4. Collect evaluation metrics: mAP50, mAP50-95, Precision, Recall  
5. Perform statistical analysis and visualize results  
6. Select the best model 

##  1. Environment & Setup

In this section, we set up the environment and install the libraries needed to run the experiments.

---

The following libraries are used for training the model, running the grid search, and analyzing the results:

In [ ]:
!pip install ultralytics --quiet
!pip install scikit-learn pandas matplotlib seaborn --quiet

In [ ]:
import os
import shutil
import yaml
import json
import time
import random
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from datetime import datetime
from collections import defaultdict
from scipy import stats

from sklearn.model_selection import KFold
from ultralytics import YOLO


# Reproducibility (fix random seeds)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("Imports done")
print(f"Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## 2. Dataset Configuration

In [ ]:
DATASET_ROOT   = Path("/kaggle/input/datasets/farahgaida/dataset-windshield/dataset_windshield")  
WORK_DIR       = Path("/kaggle/working/gridsearch")
RESULTS_DIR    = Path("/kaggle/working/results")

TRAIN_IMG_DIR  = DATASET_ROOT / "train" / "images"
TRAIN_LBL_DIR  = DATASET_ROOT / "train" / "labels"
VAL_IMG_DIR    = DATASET_ROOT / "val" / "images"  
VAL_LBL_DIR    = DATASET_ROOT / "val" / "labels"

CLASS_NAMES    = ["windshield"]   
IMG_SIZE       = 512
N_FOLDS        = 3

WORK_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f" Dataset root : {DATASET_ROOT}")
print(f" Work dir : {WORK_DIR}")
print(f" Number of folds : {N_FOLDS}")

In [ ]:
#  Merge train and validation sets

def collect_pairs(img_dir, lbl_dir):
    """Collect image and label pairs."""
    pairs = []

    for img_path in sorted(img_dir.glob("*.*")):
        if img_path.suffix.lower() not in {".jpg", ".jpeg", ".png", ".webp"}:
            continue

        lbl_path = lbl_dir / f"{img_path.stem}.txt"

        if lbl_path.exists():
            pairs.append((img_path, lbl_path))

    return pairs


train_pairs = collect_pairs(TRAIN_IMG_DIR, TRAIN_LBL_DIR)
val_pairs   = collect_pairs(VAL_IMG_DIR, VAL_LBL_DIR)
all_pairs   = train_pairs + val_pairs

print(f"Train set size   : {len(train_pairs)} images")
print(f"Validation size  : {len(val_pairs)} images")
print(f"Total pool size  : {len(all_pairs)} images")

assert len(all_pairs) > 0, "No image-label pairs found "

In [ ]:
#  Dataset overview visualization 

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# -----------------------------
# Train / Val / Pool distribution
# -----------------------------
axes[0].bar(
    ["Train", "Val", "Pool"],
    [len(train_pairs), len(val_pairs), len(all_pairs)]
)

axes[0].set_title("Dataset distribution")
axes[0].set_ylabel("Number of images")

for bar, val in zip(axes[0].patches, [len(train_pairs), len(val_pairs), len(all_pairs)]):
    axes[0].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 1,
        str(val),
        ha="center",
        va="bottom"
    )

# Fold sizes
fold_sizes = [len(all_pairs) // N_FOLDS] * N_FOLDS
for i in range(len(all_pairs) % N_FOLDS):
    fold_sizes[i] += 1

axes[1].bar(
    [f"Fold {i+1}" for i in range(N_FOLDS)],
    fold_sizes
)

axes[1].set_title(f"Fold size distribution (K={N_FOLDS})")
axes[1].set_ylabel("Number of images")


plt.tight_layout()

plt.savefig(
    RESULTS_DIR / "dataset_overview.png")

plt.show()

## 3. Grid Search Space

We test different hyperparameters that can affect YOLOv8 performance:

- **Model size**: e.g. YOLOv8n vs YOLOv8s 
- **Optimizer**: SGD, AdamW 
- **Learning rate**: initial learning step  
- **Image size**: input resolution affecting accuracy and speed 
- **Batch size**: number of images processed per iteration 
- **Regularization (weight decay)**: controls overfitting

The goal is to find a good balance between performance and training cost, especially since the dataset is relatively small.

Other training parameters (epochs, patience, etc.) are kept fixed during the search.

In [ ]:
# Hyperparameter grid

PARAM_GRID = {
    # Model 
    "model": ["yolov8n.pt", "yolov8s.pt"],

    # Optimizer 
    "optimizer": ["SGD", "AdamW"],

    # Learning rate 
    "lr0": [0.001, 0.005],

    # Weight decay (regularization)
    "weight_decay": [0.0005, 0.001],

    # Image size
    "imgsz": [512, 640],

    # Batch size
    "batch": [8, 16],
}

# Fixed parameters
FIXED_PARAMS = {
    "epochs": 40,          
    "patience": 12,        

    "device": 0,
    "verbose": False,
    "plots": False,
    "save": True,

    "seed": SEED,
    "workers": 2,
}

# Generate all combinations
keys = list(PARAM_GRID.keys())
values = list(PARAM_GRID.values())
configs = [dict(zip(keys, combo)) for combo in itertools.product(*values)]

total_runs = len(configs) * N_FOLDS

print(f"Number of configs: {len(configs)}")
print(f"Folds : {N_FOLDS}")
print(f"Total training runs : {total_runs}")
print()

print("Hyperparameter configs:")
for i, c in enumerate(configs):
    print(f"  Config {i+1:02d}: {c}")

## 4. K-Fold Setup

In [ ]:
def build_fold_dataset(fold_idx, train_indices, val_indices, all_pairs, fold_root):
    """
    Create a temporary dataset for one fold.
    Returns the path to data.yaml.
    """

    fold_dir = fold_root / f"fold_{fold_idx}"

    for split, indices in [("train", train_indices), ("val", val_indices)]:
        img_dir = fold_dir / split / "images"
        lbl_dir = fold_dir / split / "labels"

        img_dir.mkdir(parents=True, exist_ok=True)
        lbl_dir.mkdir(parents=True, exist_ok=True)

        for idx in indices:
            img_src, lbl_src = all_pairs[idx]

            shutil.copy2(img_src, img_dir / img_src.name)
            shutil.copy2(lbl_src, lbl_dir / lbl_src.name)

    # Create YAML file for YOLO
    yaml_path = fold_dir / "data.yaml"

    yaml_data = {
        "path": str(fold_dir),
        "train": "train/images",
        "val": "val/images",
        "nc": len(CLASS_NAMES),
        "names": CLASS_NAMES,
    }

    with open(yaml_path, "w") as f:
        yaml.dump(yaml_data, f, default_flow_style=False)

    return yaml_path


def cleanup_fold(fold_idx, fold_root):
    """Delete fold folder to save space"""
    fold_dir = fold_root / f"fold_{fold_idx}"
    if fold_dir.exists():
        shutil.rmtree(fold_dir)


def train_and_evaluate(config, yaml_path, run_name):
    """
    Train model with given config and return validation results.
    """

    model = YOLO(config["model"])

    train_params = {
        **FIXED_PARAMS,
        "data": str(yaml_path),

        # Hyperparameters from grid search
        "optimizer": config["optimizer"],
        "lr0": config["lr0"],
        "weight_decay": config.get("weight_decay", 0.0),
        "imgsz": config.get("imgsz", IMG_SIZE),
        "batch": config.get("batch", 8), 

        "project": str(WORK_DIR / "runs"),
        "name": run_name,
    }

    # training
    t0 = time.time()
    results = model.train(**train_params)
    duration = time.time() - t0

    # get metrics
    metrics = results.results_dict

    return {
        "mAP50": float(metrics.get("metrics/mAP50(B)", 0)),
        "mAP50_95": float(metrics.get("metrics/mAP50-95(B)", 0)),
        "precision": float(metrics.get("metrics/precision(B)", 0)),
        "recall": float(metrics.get("metrics/recall(B)", 0)),
        "duration_s": round(duration, 1),
    }

print("K-Fold functions ready")

## 5. Grid Search Execution

In this section, we run the grid search using K-Fold cross-validation.  
Each configuration is trained on all folds, and the results are collected to compare performance and find the best setup.

In [ ]:
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

indices = list(range(len(all_pairs)))
fold_splits = list(kf.split(indices))

all_results = []
fold_root = WORK_DIR / "folds"

total = len(configs) * N_FOLDS
current = 0
gs_start = time.time()

print("=" * 70)
print(f" GRID SEARCH — {len(configs)} configs × {N_FOLDS} folds = {total} runs")
print("=" * 70)

for cfg_idx, config in enumerate(configs):

    config_label = (
        f"cfg{cfg_idx+1:02d}_"
        f"{config['model'].replace('.pt','')}_"
        f"{config['optimizer']}_"
        f"lr{config['lr0']}"
    )

    fold_metrics = []

    print(f"\n{'─'*70}")
    print(f"CONFIG {cfg_idx+1}/{len(configs)}: {config}")
    print(f"{'─'*70}")

    for fold_idx, (train_idx, val_idx) in enumerate(fold_splits):

        current += 1
        run_name = f"{config_label}_fold{fold_idx+1}"

        print(
            f"Fold {fold_idx+1}/{N_FOLDS} "
            f"[{current}/{total}] "
            f"— train: {len(train_idx)} | val: {len(val_idx)}"
        )

        # create dataset for this fold
        yaml_path = build_fold_dataset(
            fold_idx,
            [indices[i] for i in train_idx],
            [indices[i] for i in val_idx],
            all_pairs,
            fold_root
        )

        # train and evaluate
        try:
            metrics = train_and_evaluate(config, yaml_path, run_name)

        except Exception as e:
            print(f" Fold {fold_idx+1} failed: {e}")
            metrics = {
                "mAP50": 0,
                "mAP50_95": 0,
                "precision": 0,
                "recall": 0,
                "duration_s": 0
            }

        # save results
        record = {
            "config_idx": cfg_idx + 1,
            "config_label": config_label,
            "model": config["model"],
            "optimizer": config["optimizer"],
            "lr0": config["lr0"],
            "batch": config["batch"],
            "imgsz": config["imgsz"],
            "weight_decay": config["weight_decay"],
            "fold": fold_idx + 1,
            **metrics
        }

        all_results.append(record)
        fold_metrics.append(metrics["mAP50"])

        print(
            f"mAP50={metrics['mAP50']:.4f} | "
            f"mAP50-95={metrics['mAP50_95']:.4f} | "
            f"P={metrics['precision']:.4f} | "
            f"R={metrics['recall']:.4f} | "
            f"time={metrics['duration_s']}s"
        )

        # delete fold data after use
        cleanup_fold(fold_idx, fold_root)

    # save intermediate results
    pd.DataFrame(all_results).to_csv(
        RESULTS_DIR / "gridsearch_checkpoint.csv",
        index=False
    )

    print(
        f"→ Config mean mAP50: {np.mean(fold_metrics):.4f} "
        f"± {np.std(fold_metrics):.4f}"
    )

# final summary 
total_time = (time.time() - gs_start) / 60

print("\n" + "=" * 70)
print(f" Grid search done in  {total_time:.1f} minutes")
print("=" * 70)

## 6. Results Aggregation and Analysis

This section collects the results from all grid search runs and analyzes them to see how the different hyperparameter choices affect performance.  
The goal is to find the most stable and best-performing configuration based on cross-validation results.

In [ ]:
df_raw = pd.DataFrame(all_results)
df_raw.to_csv(RESULTS_DIR / "gridsearch_raw.csv", index=False)

print("Grid search results (all configs × folds):")
display(df_raw.round(4))

In [ ]:
# Aggregate results per configuration (mean + std over folds)

group_cols  = ["config_idx", "model", "optimizer", "lr0", "batch", "imgsz", "weight_decay"]
metric_cols = ["mAP50", "mAP50_95", "precision", "recall"]

agg_dict = {}

for m in metric_cols:
    agg_dict[f"{m}_mean"] = (m, "mean")
    agg_dict[f"{m}_std"]  = (m, "std")

df_agg = df_raw.groupby(group_cols).agg(**agg_dict).reset_index()

df_agg = df_agg.sort_values("mAP50_mean", ascending=False).reset_index(drop=True)
df_agg["rank"] = df_agg.index + 1

df_agg.to_csv(RESULTS_DIR / "gridsearch_aggregated.csv", index=False)

print("Aggregated results (sorted by mAP50 ):")
display(df_agg.round(4))

In [ ]:
# Select best configuration (top ranked)

best = df_agg.iloc[0]

print("=" * 50)
print("Best configuration")
print("=" * 50)

print(f"Model       : {best['model']}")
print(f"Optimizer   : {best['optimizer']}")
print(f"lr0         : {best['lr0']}")
print(f"Batch size  : {best['batch']}")
print(f"Image size  : {best['imgsz']}")
print(f"Weight decay: {best['weight_decay']}")

print(f"\nmAP50       : {best['mAP50_mean']:.4f} ± {best['mAP50_std']:.4f}")
print(f"mAP50-95    : {best['mAP50_95_mean']:.4f} ± {best['mAP50_95_std']:.4f}")
print(f"Precision   : {best['precision_mean']:.4f} ± {best['precision_std']:.4f}")
print(f"Recall      : {best['recall_mean']:.4f} ± {best['recall_std']:.4f}")

print("=" * 50)

# Save best config for next notebook
best_config = {
    "model": best["model"],
    "optimizer": best["optimizer"],
    "lr0": float(best["lr0"]),
    "batch": int(best["batch"]),
    "imgsz": int(best["imgsz"]),
    "weight_decay": float(best["weight_decay"]),

    "mAP50_mean": float(best["mAP50_mean"]),
    "mAP50_std": float(best["mAP50_std"]),
}

with open(RESULTS_DIR / "best_config.json", "w") as f:
    json.dump(best_config, f, indent=2)

print("\n best_config.json saved ")

## 7. Visualizations

This section shows the grid search results through plots to better understand how different hyperparameters affect model performance and stability.

In [ ]:
# figure 1: mAP50 heatmap (optimizer vs lr0 for each model)

n_models = df_agg["model"].nunique()

fig, axes = plt.subplots(1, n_models, figsize=(7 * n_models, 5))

if n_models == 1:
    axes = [axes]

for ax, (model_name, group) in zip(axes, df_agg.groupby("model")):

    pivot = group.pivot_table(
        index="optimizer",
        columns="lr0",
        values="mAP50_mean",
        aggfunc="mean"
    )

    sns.heatmap(
        pivot,
        ax=ax,
        annot=True,
        fmt=".4f",
        cmap="YlOrRd",
        vmin=df_agg["mAP50_mean"].min() * 0.9,
        linewidths=0.5,
        cbar_kws={"label": "Mean mAP50"}
    )

    ax.set_title(f"Model: {model_name}")
    ax.set_xlabel("Learning rate (lr0)")
    ax.set_ylabel("Optimizer")

plt.suptitle(
    "mAP50 heatmap (grid search results)",
    fontsize=15,
    fontweight="bold",
    y=1.02
)

plt.tight_layout()

plt.savefig(
    RESULTS_DIR / "heatmap_mAP50.png",
    dpi=150,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# figure 2: Mean mAP50 with std for each configuration

fig, ax = plt.subplots(figsize=(14, 6))

df_plot = df_agg.copy()
df_plot["label"] = df_plot.apply(
    lambda r: f"{r['model'].replace('.pt','')}\n{r['optimizer']}\nlr={r['lr0']}",
    axis=1
)

bars = ax.bar(
    df_plot["label"],
    df_plot["mAP50_mean"],
    yerr=df_plot["mAP50_std"],
    capsize=5,
)

# values on top of bars
for bar, mean, std in zip(bars, df_plot["mAP50_mean"], df_plot["mAP50_std"]):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + std + 0.003,
        f"{mean:.4f}",
        ha="center",
        va="bottom",
        fontsize=8
    )

ax.set_title("mAP50 per configuration")

ax.set_ylabel("mAP50")
ax.set_xlabel("config")
ax.tick_params(axis="x")


ax.grid(axis="y", alpha=0.4)

plt.tight_layout()

plt.savefig(
    RESULTS_DIR / "bar_mAP50.png",
    dpi=150,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# figure 3: mAP50 distribution per fold (Top 5 configs)

# Select Top-5 configurations
top5_labels = df_agg.head(5)["config_idx"].tolist()
df_top5 = df_raw[df_raw["config_idx"].isin(top5_labels)].copy()

df_top5["label"] = df_top5.apply(
    lambda r: f"cfg{int(r['config_idx']):02d}\n"
              f"{r['model'].replace('.pt','')}\n"
              f"{r['optimizer']} lr={r['lr0']}",
    axis=1
)

fig, ax = plt.subplots(figsize=(13, 6))

df_top5.boxplot(
    column="mAP50",
    by="label",
    ax=ax
)

ax.set_title("mAP50 Distribution per Fold (Top 5 configs)")

ax.set_xlabel("config")
ax.set_ylabel("mAP50 per fold")

plt.suptitle("")  # remove default title

ax.grid(axis="y", alpha=0.4)

plt.tight_layout()

plt.savefig(
    RESULTS_DIR / "boxplot_top5.png",
    dpi=150,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# figure 4: impact of some hyperparameters on mAP50

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

params_to_plot = [
    ("model", "Model"),
    ("optimizer", "Optimizer"),
    ("lr0", "Learning Rate")
]

for ax, (param, label) in zip(axes, params_to_plot):

    grouped = df_raw.groupby(param)["mAP50"].agg(["mean", "std", "count"]).reset_index()
    grouped.columns = [param, "mean", "std", "count"]

    bars = ax.bar(
        grouped[param].astype(str),
        grouped["mean"],
        yerr=grouped["std"]
    )

    # value annotations
    for bar, mean, std in zip(bars, grouped["mean"], grouped["std"]):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + std * 0.5 + 0.001,
            f"{mean:.4f}",
            ha="center",
            va="bottom"
        )

    ax.set_title(f"Impact of {label}")

    ax.set_ylabel("Mean mAP50" if label == "Model" else "")
    ax.set_xlabel(label)

    ax.grid(axis="y", alpha=0.4)

plt.suptitle(  "Marginal Impact of Hyperparameters on mAP50")

plt.tight_layout()

plt.savefig(
    RESULTS_DIR / "marginal_impact.png",
    dpi=150,
    bbox_inches="tight"
)

plt.show()

## 8. Statistical Analysis - Friedman Test

To check if the differences between configurations are meaningful, we use the Friedman test.

This test compares multiple models evaluated on the same folds and tells us if the performance differences are statistically significant.

If the p-value is lower than 0.05, it means that the differences are unlikely to be due to randomness.

This helps us avoid choosing a model based only on small random differences.

In [ ]:
# Friedman test
pivot_stats = df_raw.pivot_table(
    index="fold",
    columns="config_idx",
    values="mAP50"
)

friedman_stat, friedman_p = stats.friedmanchisquare(
    *[pivot_stats[c].values for c in pivot_stats.columns]
)

print("=" * 50)
print("Friedman test results:")
print("=" * 50)

print(f"statistic = {friedman_stat:.4f}")
print(f"p-value : {friedman_p:.6f}")
print()

if friedman_p < 0.05:
    print("There is a significant difference between configurations.")
else:
    print("No significant difference between configurations.")
    print("So we can choose a simpler model ( YOLOv8n).")

print("=" * 50)

## 9. Final Report — Summary Table

This section summarizes the results of the grid search.

It shows the performance of all tested configurations using cross-validation, making it easier to compare them and identify the best model.

This table is used to support the final choice of the model based on both performance and stability.

In [ ]:
print("\n--- Final report ---\n")

print(f"Dataset           : {len(all_pairs)} images (train + val)")
print(f"K-Folds           : {N_FOLDS}")
print(f"Configs tested    : {len(configs)}")
print(f"Total runs        : {len(configs) * N_FOLDS}")
print()

print("BEST CONFIGURATION:")
print(f"Model : {best['model']}")
print(f"Optimizer : {best['optimizer']}")
print(f"lr0 : {best['lr0']}")
print(f"mAP50 : {best['mAP50_mean']:.4f} ± {best['mAP50_std']:.4f}")
print(f"mAP50-95 : {best['mAP50_95_mean']:.4f} ± {best['mAP50_95_std']:.4f}")
print(f"Precision : {best['precision_mean']:.4f} ± {best['precision_std']:.4f}")
print(f"Recall : {best['recall_mean']:.4f} ± {best['recall_std']:.4f}")
print()

print(f" Friedman test: χ²={friedman_stat:.4f}, p={friedman_p:.6f}")
sig = "significant" if friedman_p < 0.05 else "NOT significant"
print(f"Differences between configurations are {sig}")

print()

print(" Generated files:")
for f in sorted(RESULTS_DIR.glob("*")):
    print(f"{f.name}")

print()
